In [16]:
pip install mystic

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [mystic]2m3/4 [mystic]
Note: you may need to restart the kernel to use updated packages.


In [1]:
float('PB_M')

ValueError: could not convert string to float: 'PB_M'

In [3]:
from z3 import Int, Solver, sat, Not

i, j = Int('i'), Int('j')
size_A = 100
solver = Solver()
solver.add(i >= 0, j >= 0)
solver.add(Not(j + 1 == 0))          # 避免除零
solver.add(i / (j + 1) >= 0, i / (j + 1) < size_A)

if solver.check() == sat:
    model = solver.model()
    print(model)
    U_i = model[i].as_long() + 1
    U_j = model[j].as_long() + 1
    print(f"Safe bounds: i ∈ [0, {U_i}], j ∈ [0, {U_j}]")
else:
    print("No feasible bounds.")

[i = 0, j = 0, div0 = [else -> 0], mod0 = [else -> 0]]
Safe bounds: i ∈ [0, 1], j ∈ [0, 1]


In [ ]:
from z3 import Int, Solver, sat, o

i, j = Int('i'), Int('j')
solver = Solver()
solver.add(i >= 0, i < 10)
solver.add(j >= 0, j < 10)
solver.add(i**2 + j >= 0, i**2 + j < 100)  # 0 <= i^2 + j < 100

# Check feasibility
result = solver.check()
print("Feasible?", "Yes" if result == sat else "No")

# Print a solution if exists
if result == sat:
    model = solver.model()
    print(f"Example solution: i = {model[i]}, j = {model[j]}")

Feasible? Yes
Example solution: i = 4, j = 9


In [15]:
import sympy as sp

x, y = sp.symbols('x y')
inequality = sp.Or(sp.And(x**2 + y**2 >= 1, x >= 0, y >= 0), sp.And(x**2 + y**2 <= 1, x <= 0, y <= 0))

solution = sp.solve(inequality)

print(solution)

TypeError: unsupported operand type(s) for -: 'Or' and 'int'

In [ ]:
from pulp import LpMaximize, LpProblem, LpVariable, lpSum

# 创建问题实例
prob = LpProblem("Nonlinear_Inequality_System", LpMaximize)

# 定义变量（整数）
x = LpVariable("x", lowBound=0, upBound=5, cat='Integer')
y = LpVariable("y", lowBound=0, upBound=5, cat='Integer')

# 引入辅助变量 z = x*y（假设 x 和 y 的上界已知）
z = LpVariable("z", lowBound=0, upBound=25, cat='Integer')

# 添加线性化约束
# z ≤ M1*y （M1 是 x 的上界）
prob += z <= 5 * y
# z ≤ M2*x （M2 是 y 的上界）
prob += z <= 5 * x
# z ≥ x + y - 1
prob += z >= x + y - 1
# z ≤ x + (M1-1)*y
prob += z <= x + 4 * y

# 添加其他约束
prob += x + y >= 3
prob += x**2 + y**2 <= 20  # 注：PuLP 可接受简单二次项，但复杂非线性需线性化

# 设置目标函数（这里仅为示例，可根据需求修改）
prob += x + y

# 求解问题
prob.solve()

# 输出结果
if prob.status == 1:  # Optimal
    print("找到可行解：")
    print(f"x = {int(x.value())}, y = {int(y.value())}, z = {int(z.value())}")
else:
    print("没有找到可行解。")

TypeError: unsupported operand type(s) for ** or pow(): 'LpVariable' and 'int'

In [1]:
from constraint import Problem

problem = Problem()

# 添加变量和可能的值范围
problem.addVariable('x', range(-5, 6))
problem.addVariable('y', range(-5, 6))

# 添加约束条件
problem.addConstraint(lambda x, y: x**2 + y**2 <= 25, ('x', 'y'))
problem.addConstraint(lambda x, y: x + y >= 3, ('x', 'y'))
problem.addConstraint(lambda x, y: y - x < 4, ('x', 'y'))

solutions = problem.getSolutions()
print(solutions)

[{'x': 5, 'y': 0}, {'x': 4, 'y': 3}, {'x': 4, 'y': 2}, {'x': 4, 'y': 1}, {'x': 4, 'y': -1}, {'x': 4, 'y': 0}, {'x': 3, 'y': 4}, {'x': 3, 'y': 3}, {'x': 3, 'y': 2}, {'x': 3, 'y': 1}, {'x': 3, 'y': 0}, {'x': 2, 'y': 4}, {'x': 2, 'y': 3}, {'x': 2, 'y': 2}, {'x': 2, 'y': 1}, {'x': 1, 'y': 4}, {'x': 1, 'y': 3}, {'x': 1, 'y': 2}, {'x': 0, 'y': 3}]


In [54]:
from z3 import Int, sat, Optimize

size_A1, size_A2 = 80, 90
i_bounds=(0, size_A1)
j_bounds=(0, size_A2)

# 初始化变量
i, j = Int('i'), Int('j')

# 1. 设置循环变量范围
L_i, U_i = i_bounds
L_j, U_j = j_bounds

A_1 = i + j + i*j
A_2 = i*i

idx = i*j*((size_A1-i)**2+(size_A2-(j/2))**2)

# 最大化 idx
opt_max = Optimize()
opt_max.add(i >= L_i, i <= U_i, j >= L_j, j <= U_j, A_1 <= size_A1, A_2 <= size_A2, A_1 >=0, A_2 >= 0)
opt_max.maximize(idx)
if opt_max.check() == sat:
    print(opt_max.model())
    print(opt_max.model().eval(A_1), opt_max.model().eval(A_2))
else:
    print("未知")



# 最小化 idx
# opt_min = Optimize()
# opt_min.add(i >= L_i, i <= U_i, j >= L_j, j <= U_j)
# opt_min.minimize(idx)
# if opt_min.check() == sat:
#     min_val = opt_min.model().eval(idx).as_long()
# else:
#     min_val = "未知"

# print(f"第二维下标表达式范围: [{min_val}, {max_val}]")

[i = 8, j = 8]
80 64


In [7]:
from z3 import Int, Solver, sat, Or, Optimize, IntVal

def analyze_array_access(size_A1, size_A2, i_bounds=(0, 10), j_bounds=(0, 10)):
    # 初始化变量
    i, j = Int('i'), Int('j')
    
    # 1. 设置循环变量范围
    L_i, U_i = i_bounds
    L_j, U_j = j_bounds
    
    # 2. 计算第一维范围（直接等于循环边界）
    print(f"第一维下标 i 的范围: [{L_i}, {U_i}]")
    
    # 3. 计算第二维下标的极值
    idx = -i - j + i*j - j*j  # 下标表达式
    
    # 最大化 idx
    opt_max = Optimize()
    opt_max.add(i >= L_i, i <= U_i, j >= L_j, j <= U_j)
    opt_max.maximize(idx)
    if opt_max.check() == sat:
        max_val = opt_max.model().eval(idx).as_long()
    else:
        max_val = "未知"
    
    # 最小化 idx
    opt_min = Optimize()
    opt_min.add(i >= L_i, i <= U_i, j >= L_j, j <= U_j)
    opt_min.minimize(idx)
    if opt_min.check() == sat:
        min_val = opt_min.model().eval(idx).as_long()
    else:
        min_val = "未知"
    
    print(f"第二维下标表达式范围: [{min_val}, {max_val}]")
    
    # 4. 验证越界风险
    if isinstance(max_val, int) and max_val >= size_A2:
        print("警告: 第二维下标最大值可能越界!")
    if isinstance(min_val, int) and min_val < 0:
        print("警告: 第二维下标最小值可能为负!")

# 示例调用
size_A1, size_A2 = 10, 20
analyze_array_access(
    size_A1=size_A1,
    size_A2=size_A2,
    i_bounds=(-size_A1, size_A1),
    j_bounds=(-size_A2, size_A2)
)

第一维下标 i 的范围: [-10, 10]
第二维下标表达式范围: [-610, 34]
警告: 第二维下标最大值可能越界!
警告: 第二维下标最小值可能为负!


In [2]:
import random
from itertools import combinations_with_replacement

def generate_quadratic_access_functions(
    depth_stmt,           # 循环嵌套深度（变量数）
    depth_array_write,     # 需要生成的访问函数数量
    arg_bounds_coef=1,     # 系数取值范围（如 -1, 0, 1）
    max_degree=2,          # 最高次数（1=线性，2=二次）
    enable_multi_terms=False,  # 是否允许多基项
    max_terms_per_func=3   # 每个函数最多基项数（若启用多基项）
):
    # 基项生成：常数项 + 线性项 + 二次项
    variables = range(depth_stmt)
    terms = [()]  # 常数项
    terms.extend([(v,) for v in variables])  # 线性项
    
    if max_degree >= 2:
        terms.extend(combinations_with_replacement(variables, 2))  # 二次项
    
    # 系数取值范围和概率分布
    value_consts = range(-arg_bounds_coef, arg_bounds_coef + 1)
    prob_conts = [1/(3*(2*arg_bounds_coef))] * len(value_consts)
    prob_conts[arg_bounds_coef] = 2/3  # 0的概率为2/3
    
    # 基项权重调整：高阶项概率降为低阶的2/3
    term_weights = [1.0] * len(terms)
    for i, term in enumerate(terms):
        if len(term) == 2:  # 二次项
            term_weights[i] *= 2/3
    
    # 生成访问函数
    array_write_access_functions = []
    
    for _ in range(depth_array_write):
        access_function = [0] * len(terms)
        selected_indices = []
        remaining_weights = term_weights.copy()
        continue_prob = 1.0  # 初始继续概率为1 (2/3)^0

        while True:
            # 决定是否继续选择基项
            if random.random() > continue_prob:
                break  # 终止选择

            # 选择基项（排除已选项）
            valid_indices = [i for i in range(len(terms)) if remaining_weights[i] > 0]
            if not valid_indices:
                break

            idx = random.choices(valid_indices, weights=[remaining_weights[i] for i in valid_indices])[0]
            selected_indices.append(idx)
            remaining_weights[idx] = 0  # 禁用已选项

            # 衰减继续概率：(2/3)^n
            continue_prob *= 2/3

        # 为选中的基项分配系数
        for idx in selected_indices:
            coef = random.choices(value_consts, prob_conts)[0]
            access_function[idx] = coef

        array_write_access_functions.append(access_function)
    
    return array_write_access_functions, terms

def symbolic_expression(func_coefs, terms, array_name='A'):
    """将系数向量转换为可读的数组访问表达式"""
    vars = ['i','j','k','l','m','n']  # 变量名池
    parts = []
    
    for coef, term in zip(func_coefs, terms):
        if coef == 0:
            continue
        
        # 构造项表达式
        term_str = ''
        if term:  # 非常数项
            term_str = '*'.join([vars[v] for v in term])
        
        # 添加系数
        if coef == 1 and term:
            parts.append(f"{term_str}")
        elif coef == -1 and term:
            parts.append(f"-{term_str}")
        elif not term: # 常数项
            parts.append(f"{coef}")
        else:
            parts.append(f"{coef}*{term_str}")
    
    if not parts:
        return f"{array_name}[0]"
    
    expr = ' + '.join(parts).replace('+ -', '- ')
    return f"{array_name}[{expr}]"

# 示例调用
depth_stmt = 3       # 3层循环（变量i,j,k）
depth_array_write = 5 # 生成5个访问函数
arg_bounds_coef = 1   # 系数范围：-1, 0, 1

access_funcs, basis = generate_quadratic_access_functions(
    depth_stmt,
    depth_array_write,
    arg_bounds_coef=arg_bounds_coef,
    max_degree=2,
    enable_multi_terms=True,
    max_terms_per_func=2
)

# 打印符号化表达式
print("生成的数组访问表达式：")
for i, func in enumerate(access_funcs):
    expr = symbolic_expression(func, basis, array_name='X')
    print(f"{i+1}: {expr}")

# 打印原始系数矩阵
print(basis)
print("\n对应的系数矩阵（每行一个函数）：")
for row in access_funcs:
    print(row)

生成的数组访问表达式：
1: X[0]
2: X[0]
3: X[0]
4: X[1]
5: X[-k + i*i]
[(), (0,), (1,), (2,), (0, 0), (0, 1), (0, 2), (1, 1), (1, 2), (2, 2)]

对应的系数矩阵（每行一个函数）：
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, -1, 1, 0, 0, 0, 0, 0]


In [ ]:
range(2, 5), \
range(1, 5), \
range(1, 4), \
range(2, 7, 2), \
range(1, 4), \
range(1, 6, 2), \
range(1, 3), \
range(1, 4), \
range(1, 3), \
range(2, 7, 2), \
range(10)

In [1]:
(5-2)*(5-1)*(4-1)*(8-2)/2*(4-1)*(7-1)/2*(3-1)*(4-1)*(3-1)*(8-2)/2*10

349920.0

In [ ]:
import random
import math
from collections import deque

class TreeNode:
    def __init__(self, val):
        self.val = val
        self.children = []
        self.parent = None
        self.a = False  # 是否获得属性 'a'
        self.actual_prob = 0.0  # 实际计算概率
        self.groups = None  # 存储分组结果

    def add_child(self, child):
        child.parent = self
        self.children.append(child)

def calculate_weighted_prob(child_count, base_prob=0.5):
    """基于子节点数量计算加权概率"""
    if child_count == 0:
        return 0.0  # 没有子节点则不计算概率
    weight = math.log1p(child_count - 1) / math.log1p(5)  # 对数加权
    weighted_prob = base_prob + (1 - base_prob) * (1 - 1 / (1 + weight))
    return min(weighted_prob, 0.99)  # 上限 0.99

def split_children_into_groups(children):
    """将子节点分成连续的3组（组1非空）"""
    if not children:
        return None
    
    # 随机选择两个分割点（确保组1至少有一个节点）
    split1 = random.randint(1, len(children))  # 组1的结束位置（至少1个）
    split2 = random.randint(split1, len(children))  # 组2的结束位置（可以=split1）
    
    return {
        "group1": children[:split1],
        "group2": children[split1:split2],
        "group3": children[split2:]
    }

def propagate_attribute_iterative(root, initial_prob=0.5, decay_factor=0.25):
    """迭代方式传播属性 'a'（新概率规则）"""
    stack = deque()
    post_stack = deque()  # 用于后序遍历
    
    # 初始：所有节点入栈（但只有有子节点的节点会计算概率）
    stack.append((root, initial_prob, 1.0))
    
    # 第一阶段：DFS 遍历，计算概率
    while stack:
        node, current_prob, parent_prob = stack.pop()
        child_count = len(node.children)
        
        if child_count > 0:
            # 计算基础概率（基于子节点数量）
            base_prob = calculate_weighted_prob(child_count, current_prob)
            # 应用父节点的影响因子
            node.actual_prob = base_prob * parent_prob
            
            if random.random() < node.actual_prob:
                node.a = True
                node.groups = split_children_into_groups(node.children)
                if node.parent:
                    # 新规则：每有一个子节点有 'a'，父节点概率乘以衰减系数
                    successful_children = sum(1 for child in node.children if child.a)
                    decayed_prob = current_prob * (decay_factor ** successful_children)
                    post_stack.append((node.parent, initial_prob, decayed_prob))
        
            # 继续处理子节点
            for child in reversed(node.children):
                stack.append((child, initial_prob, 1.0))
    
    # 第二阶段：后序遍历，处理父节点
    while post_stack:
        node, current_prob, parent_prob = post_stack.pop()
        child_count = len(node.children)
        
        if child_count > 0:
            base_prob = calculate_weighted_prob(child_count, current_prob)
            node.actual_prob = base_prob * parent_prob
            
            if random.random() < node.actual_prob:
                node.a = True
                node.groups = split_children_into_groups(node.children)
                if node.parent:
                    successful_children = sum(1 for child in node.children if child.a)
                    decayed_prob = current_prob * (decay_factor ** successful_children)
                    post_stack.append((node.parent, initial_prob, decayed_prob))

def print_tree_with_details(node, level=0):
    """打印树结构（带分组信息）"""
    prefix = "  " * level
    attr = " [a]" if node.a else ""
    prob = f" (Prob: {node.actual_prob:.4f}, Children: {len(node.children)})"
    print(f"{prefix}{node.val}{attr}{prob}")
    
    if node.a and node.groups:
        for group_name, group_nodes in node.groups.items():
            print(f"{prefix}  {group_name}: {[n.val for n in group_nodes]}")
    
    for child in node.children:
        print_tree_with_details(child, level + 1)

# 示例用法
if __name__ == "__main__":
    root = TreeNode("Root")
    
    # 构建一个深层树（测试递归问题）
    current = root
    for i in range(1, 5):  # 深层树（1000层）
        child = TreeNode(f"Node_{i}")
        current.add_child(child)
        current = child
    
    # 运行迭代版本的传播算法
    propagate_attribute_iterative(root)
    
    # 打印部分结果（避免输出太长）
    print("树结构（部分）及属性 'a' 的分布:")
    current = root
    for _ in range(5):  # 只打印前10层
        attr = " [a]" if current.a else ""
        prob = f" (Prob: {current.actual_prob:.4f})"
        print("  " * _ + current.val + attr + prob)
        if not current.children:
            break
        current = current.children[0]
        
    # 构建一个更大的示例树
    # root = TreeNode("Root")
    
    # # 第一层子节点
    # for i in range(1, 4):
    #     child = TreeNode(f"Child{i}")
    #     root.add_child(child)
        
    #     # 第二层子节点
    #     for j in range(1, i+2):  # 每个子节点的子节点数量递增
    #         grandchild = TreeNode(f"Grandchild{i}-{j}")
    #         child.add_child(grandchild)
            
    #         # 第三层子节点
    #         if j % 2 == 0:
    #             great_grandchild = TreeNode(f"GreatGrandchild{i}-{j}")
    #             grandchild.add_child(great_grandchild)
    
    # # 执行属性传播算法
    # propagate_attribute_iterative(root)
    
    # # 打印结果
    # print("树结构及属性'a'的分布(考虑子节点数量加权):")
    # print_tree_with_details(root)
    

树结构（部分）及属性 'a' 的分布:
Root (Prob: 0.5000)
  Node_1 (Prob: 0.2500)
    Node_2 [a] (Prob: 0.5000)
      Node_3 (Prob: 0.5000)
        Node_4 (Prob: 0.0000)


In [5]:
import random
import math
from collections import deque
import numpy as np

class TreeNode:
    def __init__(self, val):
        self.val = val
        self.children = []
        self.parent = None
        self.flag = False  # 是否获得flag
        self.actual_prob = 0.0  # 实际计算概率
        self.processed = False  # 标记是否已处理

    def add_child(self, child):
        child.parent = self
        self.children.append(child)

def calculate_weighted_prob(child_count, base_prob=0.5):
    """计算加权概率（基于子节点数量）"""
    # if child_count == 0:
    return base_prob
    # weight = math.log1p(child_count)  # 使用log1p避免child_count=0的情况
    # weighted_prob = base_prob * (1 + weight) / (1 + math.log1p(5))  # 标准化到合理范围
    # return min(weighted_prob, 0.99)  # 上限0.99

def propagate_flags(root, initial_prob=0.5, decay=0.5):
    """完整的两阶段传播算法"""
    if not root:
        return
    
    # 第一阶段：后序遍历计算所有叶节点的初始标记
    stack = [(root, False)]
    post_order = []
    
    while stack:
        node, visited = stack.pop()
        if visited:
            post_order.append(node)
            if not node.children:  # 叶节点
                node.actual_prob = initial_prob
                node.flag = random.random() < node.actual_prob
        else:
            stack.append((node, True))
            for child in reversed(node.children):
                stack.append((child, False))
    
    # 第二阶段：从叶节点到根节点的反向传播
    for node in post_order:
        if node.children:  # 非叶节点
            # 计算基础概率（子节点概率平均值）
            node.base_prob = sum(child.actual_prob for child in node.children) / len(node.children)
            
            # 计算衰减因子（每个有flag的子节点乘以decay）
            node.decay_factor = 1.0
            for child in node.children:
                if child.flag:
                    node.decay_factor *= decay
            print(node.val, node.base_prob, node.decay_factor)
            # 计算加权概率
            node.actual_prob = calculate_weighted_prob(
                len(node.children), 
                node.base_prob * node.decay_factor
            )
            
            # 决定当前节点是否标记flag
            node.flag = random.random() < node.actual_prob

def print_tree_with_prob(node, level=0):
    """打印树结构（带概率）"""
    prefix = "  " * level
    attr = " [flag]" if node.flag else ""
    prob = f" (Prob: {node.actual_prob:.4f}, Children: {len(node.children)})"
    print(f"{prefix}{node.val}{attr}{prob}")
    for child in node.children:
        print_tree_with_prob(child, level + 1)

def build_sample_tree():
    """构建一个示例树结构"""
    root = TreeNode("Root")
    
    # 第一层
    a = TreeNode("A")
    b = TreeNode("B")
    root.add_child(a)
    root.add_child(b)
    
    # 第二层
    c = TreeNode("C")
    d = TreeNode("D")
    e = TreeNode("E")
    a.add_child(c)
    a.add_child(d)
    b.add_child(e)
    
    # 第三层（叶节点）
    f = TreeNode("F")
    g = TreeNode("G")
    h = TreeNode("H")
    i = TreeNode("I")
    c.add_child(f)
    c.add_child(g)
    d.add_child(h)
    e.add_child(i)
    
    return root

# 示例用法
if __name__ == "__main__":
    # 构建示例树
    # root = build_sample_tree()
    
    # # 运行迭代版本的传播算法
    # propagate_attribute_iterative(root)
    
    # # 打印树结构
    # print("树结构及flag分布:")
    # print_tree_with_prob(root)
    
    root = TreeNode("Root")
    # 构建一个深层树（测试递归问题）
    current = root
    for i in range(1, 10):  # 深层树（1000层）
        child = TreeNode(f"Node_{i}")
        current.add_child(child)
        current = child
    
    # 运行迭代版本的传播算法
    propagate_flags(root)
    
    # 打印部分结果（避免输出太长）
    print("树结构（部分）及flag 的分布:")
    current = root
    for _ in range(10):  # 只打印前10层
        attr = " [flag]" if current.flag else ""
        prob = f" (Prob: {current.actual_prob:.4f})"
        print("  " * _ + current.val + attr + prob)
        if not current.children:
            break
        current = current.children[0]

Node_8 0.5 0.5
Node_7 0.25 1.0
Node_6 0.25 1.0
Node_5 0.25 0.5
Node_4 0.125 1.0
Node_3 0.125 1.0
Node_2 0.125 0.5
Node_1 0.0625 1.0
Root 0.0625 1.0
树结构（部分）及flag 的分布:
Root (Prob: 0.0625)
  Node_1 (Prob: 0.0625)
    Node_2 (Prob: 0.0625)
      Node_3 [flag] (Prob: 0.1250)
        Node_4 (Prob: 0.1250)
          Node_5 (Prob: 0.1250)
            Node_6 [flag] (Prob: 0.2500)
              Node_7 (Prob: 0.2500)
                Node_8 (Prob: 0.2500)
                  Node_9 [flag] (Prob: 0.5000)
